# 05 · Forecast and promo budget optimization (MILP)

Phase 5 plans the next 13 weeks on top of the recommended regular prices from Phase 4. This is trade promotion optimization (TPO): which product to promote, in which week, with which tactic and discount, and how much budget to spend.

1. **Forecast:** weekly baseline per product = recent base volume × **seasonal index** (Fourier and holiday effects from the Bayesian model, relative to the last 13 weeks) × the regular-price response.
2. **Options:** every product × week × tactic (TPR, feature, display, feature + display) × depth (10 / 20 / 30%). For each option:
   - promo units = baseline × (1 − d)^δ × e^(tactic effect), where δ is the promo elasticity (Phase 2) and the tactic effect is measured at its usual store coverage;
   - the pantry dip removes units in the next 3 weeks;
   - trade spend = discount × regular price × promo units + an assumed fixed fee.
3. **Plan (MILP, PuLP + HiGHS):** maximise incremental profit (or revenue). Constraints:
   - trade spend ≤ budget;
   - at most 4 promos per product;
   - at most one promo per product in any 4-week window, so pantry dips never overlap;
   - weekly caps on feature and display slots.
4. **Budget:** solve at budgets from $0 up to the spend of the unconstrained plan. The recommended budget is the last step that still returns ≥ $0.25 per extra $1, within the available budget.

Two benchmarks:
- **last year's calendar**: the same weeks one year earlier, scored by the same model (business as usual);
- a **greedy** planner that ranks options by gain per dollar.

BOGO and coupons are not in the data, so they can't be measured or planned.

Logic lives in `src/promo/`; run `make promo` first.

In [ ]:
# reload edited src/ modules automatically (avoids stale imports after code changes)
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd

from src.config import (
    BUDGET_FRONTIER_PATH,
    FIGURES,
    FORECAST_BACKTEST_PATH,
    FORECAST_PATH,
    PROMO_PLAN_PATH,
    PROMO_SUMMARY_PATH,
    Config,
)
from src.plots import FigureStore, PlotStyle, PromoPlotter
from src.schema import Col

style = PlotStyle()
style.apply()
plotter, store, promo_cfg = PromoPlotter(style), FigureStore(FIGURES), Config.load().promo
forecast, plans = pd.read_parquet(FORECAST_PATH), pd.read_parquet(PROMO_PLAN_PATH)
frontier, summary = pd.read_parquet(BUDGET_FRONTIER_PATH), pd.read_parquet(PROMO_SUMMARY_PATH)

## Choose the goal
Set `objective` to `"profit"` or `"revenue"` and re-run the cells below. Each goal uses its own Phase 4 prices. KPIs always report real profit, net of trade spend.

In [ ]:
objective = "profit"  # "profit" or "revenue"

## Forecast accuracy (backtest)
Four methods forecast the weekly baseline (the top-down baseline from Phase 3) for the 13 holdout weeks, trained only on the weeks before them:

- **naive flat:** the mean of the last 13 weeks;
- **seasonal index:** that level × the Bayesian model's Fourier and holiday effects;
- **SARIMAX(0,1,1):** per product, local level plus calendar regressors;
- **XGBoost:** one model for all products, forecasting each 1–13-week-ahead ratio to the recent level.

`forecast.method: auto` uses the lowest WAPE for the horizon.

In [ ]:
backtest = pd.read_parquet(FORECAST_BACKTEST_PATH)
store.save(plotter.backtest(backtest), "05_forecast_backtest")
backtest.round(3)

In [ ]:
store.save(plotter.forecast(forecast, objective), f"05_forecast_{objective}")

## Budget frontier
Each point is the best plan the MILP finds at that budget. Returns diminish because the best options are taken first. The recommended budget is the last step whose marginal return still clears the hurdle.

In [ ]:
store.save(
    plotter.frontier(frontier, objective, promo_cfg.budget, promo_cfg.min_marginal_return),
    f"05_frontier_{objective}",
)

## Scenarios
All plans use the recommended regular prices; changes are against current prices with no promotions.
- `last_year_calendar` replays the same weeks last year (business as usual).
- `milp_at_last_year_spend` gives the MILP last year's budget; it may leave money unspent.
- `greedy_plan` and `milp_plan` are solved at the recommended budget.

In [ ]:
cols = [
    Col.SCENARIO,
    Col.TRADE_SPEND,
    Col.N_PROMOS,
    "volume_change",
    "revenue_change",
    "profit_change",
]
summary[summary[Col.OBJECTIVE] == objective][cols].round(3)

In [ ]:
store.save(plotter.scenarios(summary, objective), f"05_scenarios_{objective}")

## Promo calendar (MILP plan)
Only promoted products are shown. Shallow discounts with merchandising (feature + display) earn the most: under a 30% margin, deep TPRs buy volume but lose profit.

In [ ]:
store.save(plotter.calendar(forecast, objective), f"05_calendar_{objective}")

## Tactic mix by plan
Last year's calendar leans on deep TPRs; the optimized plans use shallow discounts with display.

In [ ]:
view = plans[plans[Col.OBJECTIVE] == objective]
view.groupby([Col.SCENARIO, Col.TACTIC, Col.DISCOUNT_DEPTH]).size().unstack(fill_value=0)

## Plan detail
Each MILP promo with its lift, incremental units after the pantry dip, trade spend and incremental profit.

In [ ]:
milp = view[view[Col.SCENARIO] == "milp_plan"]
names = forecast[[Col.UPC, Col.DESCRIPTION, Col.PRODUCT_SIZE]].drop_duplicates(Col.UPC)
cols = [
    Col.DESCRIPTION,
    Col.PRODUCT_SIZE,
    Col.WEEK,
    Col.TACTIC,
    Col.DISCOUNT_DEPTH,
    Col.LIFT,
    Col.INCREMENTAL,
    Col.TRADE_SPEND,
    Col.INCREMENTAL_PROFIT,
    Col.ROI,
]
milp.merge(names, on=Col.UPC)[cols].round(dict.fromkeys(cols[4:], 2))